# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balabhadra3141/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)

In [6]:
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Fetch HF_TOKEN from the Colab environment/secrets.
HF_TOKEN = userdata.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named 'HF_TOKEN'."
    )

# Connect to DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Authenticate DuckDB with Hugging Face
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

# Warehouse paths
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# ML-04 windows
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("HF_TOKEN fetched successfully.")
print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Outcome window: March 2026")

HF_TOKEN fetched successfully.
Connected to FlyRank warehouse.
Feature window: February 2026
Outcome window: March 2026


## 1. Unit of analysis + time window

**Unit of analysis:** One row represents one content item for one client (`client_hash_id × content_hash_id`) at the decision point.

**Feature window:** February 2026 (`2026-02-01` to `2026-02-28`). Features use only information available by the end of February.

**Label window:** March 2026 (`2026-03-01` to `2026-03-31`). The label is measured after the feature window.

**Ranking target:** Rank pages by whether they subsequently go dark in March, defined as recording zero measured GSC clicks during the March outcome window.

**Deliberate exclusion:** I exclude label-derived fields and product decision fields from the features because they would either reveal the outcome or reproduce an existing decision rule.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Unique content items:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

print("\nDate/time-like columns:")
date_cols = [
    col for col in df.columns
    if any(x in col.lower() for x in ["date", "time", "day"])
]
print(date_cols)

Dataset shape: (30000, 44)
Unique content items: 30000
Unique clients: 32

Date/time-like columns:
['days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update']


## 2. Fields: feature / label / context / excluded

### Feature candidates

I will use five observable features from the February decision window:

* `imp_feb` — February GSC impressions.
* `clk_feb` — February GSC clicks.
* `avg_position_feb` — impression-weighted average search position in February.
* `content_age_days` — age of the content at the February decision point.
* `days_since_last_update` — freshness/staleness signal available at the decision point.

### Label

* `went_dark` — 1 when measured March GSC clicks are zero, otherwise 0.

### Context

* `client_hash_id`
* `content_hash_id`

These are used for grouping and joins, not as model features.

### Excluded

* Future March performance fields — unavailable at the February decision point.
* Label-derived fields — would leak the outcome.
* `health_score`, `priority_score`, `action_type`, `refresh_tier` — these encode existing product decisions rather than independent observable signals.
* Raw client names, domains, URLs, queries, or other identifying information.

## 3. Verify it with queries (grain, counts, missing values, windows)

### Query 1 — Grain

The daily performance table should contain one row per report date, client, and content item.

In [7]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT
            report_date || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        ) AS unique_client_content_day,
        COUNT(*) -
        COUNT(DISTINCT
            report_date || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        ) AS duplicate_rows
    FROM {FEB}
""").df()

grain_check

,rows,unique_client_content_day,duplicate_rows
0,7355108,7355108,0


### Query 2 — February feature-window count and date span

This verifies the size and date coverage of the February feature window.

In [8]:
window_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {FEB}
""").df()

window_check

,rows,clients,content_items,first_date,last_date
0,7355108,54,321546,2026-02-01,2026-02-28


### Query 3 — GSC availability

GSC availability is checked explicitly with `IS TRUE` so unavailable observations are not treated as zero traffic.

In [9]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS gsc_unavailable_or_null_rows
    FROM {FEB}
""").df()

availability_check

,total_rows,gsc_available_rows,gsc_unavailable_or_null_rows
0,7355108,2621783,4733325


### Build the five-feature frame

In [10]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW feb_agg AS

SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS imp_feb,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clk_feb,

    SUM(gsc_sum_position) FILTER (
        WHERE gsc_data_available IS TRUE
    ) / NULLIF(
        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ),
        0
    ) AS avg_position_feb

FROM {FEB}

GROUP BY
    client_hash_id,
    content_hash_id

HAVING
    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) >= 100

    AND

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) >= 3
""")

In [12]:
feature_frame = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        f.imp_feb,
        f.clk_feb,
        f.avg_position_feb,

        DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) AS content_age_days,

        DATE_DIFF(
            'day',
            c.content_updated_date,
            DATE '2026-02-28'
        ) AS days_since_last_update

    FROM feb_agg f

    INNER JOIN read_parquet('{DIM_CONTENT}') c
        ON f.content_hash_id = c.content_hash_id

    WHERE
        c.is_published IS TRUE
        AND c.content_created_date <= DATE '2026-02-28'
""").df()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head()

Feature frame shape: (29700, 7)


,client_hash_id,content_hash_id,imp_feb,clk_feb,avg_position_feb,content_age_days,days_since_last_update
0,client_3ffa76342f366962,content_32bdebcb01540202,551.0,17.0,3.758621,154,-81
1,client_e547b89c05043229,content_4c1e972bec56132e,2882.0,15.0,10.403539,344,-104
2,client_e547b89c05043229,content_4e48bd81bb37eb4f,7343.0,3.0,45.407054,344,-104
3,client_e547b89c05043229,content_7e131483384291cf,6206.0,6.0,8.934096,344,-104
4,client_e547b89c05043229,content_3f12146e57e4baf0,16877.0,29.0,4.443444,344,-104


### Build the March label

In [13]:
march_label = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS measured_march_days,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS imp_mar,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clk_mar

    FROM {MAR}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("March label rows:", len(march_label))

March label rows: 331437


In [14]:
frame = feature_frame.merge(
    march_label,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Keep only pages with at least one measured March day.
frame = frame.loc[
    frame["measured_march_days"].fillna(0) > 0
].copy()

frame["imp_mar"] = frame["imp_mar"].fillna(0)
frame["clk_mar"] = frame["clk_mar"].fillna(0)

frame["went_dark"] = (
    frame["clk_mar"] == 0
).astype(int)

print("Final contract frame:", frame.shape)
print("Positive labels:", frame["went_dark"].sum())
print("Positive rate:", round(frame["went_dark"].mean(), 3))

Final contract frame: (29353, 11)
Positive labels: 1159
Positive rate: 0.039


## The leakage trap

I deliberately add a column derived directly from the March outcome. This information would not be available at the February decision point, so it should make the quick score suspiciously strong. I then remove it and keep the honest feature set.

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model_features = [
    "imp_feb",
    "clk_feb",
    "avg_position_feb",
    "content_age_days",
    "days_since_last_update"
]

X = frame[model_features].copy()
y = frame["went_dark"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

honest_pred = model.predict(X_test)

print(
    "Honest accuracy:",
    round(accuracy_score(y_test, honest_pred), 4)
)

Honest accuracy: 0.9605


In [16]:
leaky = frame[model_features].copy()

# DELIBERATE LEAK:
# This is literally the target, so it would never be available
# at the February decision point.
leaky["future_label_leak"] = frame["went_dark"]

X_leak = leaky

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)

print(
    "Leaky accuracy:",
    round(accuracy_score(y_test, leaky_pred), 4)
)

Leaky accuracy: 1.0


In [17]:
del leaky["future_label_leak"]

print("Leak removed.")
print("Final model features:")
print(model_features)

Leak removed.
Final model features:
['imp_feb', 'clk_feb', 'avg_position_feb', 'content_age_days', 'days_since_last_update']


### 4. Data limitation

The warehouse is an unbalanced panel: clients do not all have the same historical coverage, and GSC/GA4 availability varies by client and date. Therefore, a page with no measured data cannot automatically be interpreted as zero traffic. The March label is restricted to pages with measured March GSC data, which reduces this ambiguity but also reduces the available modeling universe.


### Self-check

* [x] Defined the unit of analysis and separated feature and outcome windows.
* [x] Identified the warehouse tables used for the lane.
* [x] Used exactly three verification queries.
* [x] Verified GSC availability with `IS TRUE`.
* [x] Built five features maximum.
* [x] Added a “knowable at the decision moment” explanation for each feature.
* [x] Demonstrated one deliberate label-leakage feature.
* [x] Removed the leakage before keeping the final feature set.
* [x] Documented one limitation of the data slice.
* [x] No client names, URLs, private queries, or identifying information are used.